In [ ]:
import numpy as np
from math import sqrt, atan2, pi

np.set_printoptions(
    precision=6,
    suppress=True,
    linewidth=160
)

# ============================================================
# 1. INPUT
# ============================================================
# Kolonnerekkefølge i ukjente:
# [orientering_T620, orientering_T827, dX_T620, dY_T620, dX_T827, dY_T827]
#
# Skalere til annen oppgave:
# - legg inn kjente og ukjente punkter i punkter
# - legg ukjente punkter inn i ukjente
# - legg stasjoner med orienteringsukjent inn i orienteringer
# - oppdater retning_obs og avstand_obs

punkter = {
    # "Punktnavn": [X, Y, kjent?]

    "A": [-885.942,   39.263,  True],
    "B": [0.000,      0.000,  True],
    "C": [-188.967, 1331.959, True],
    "D": [-819.168, 1315.756, True],

    "1": [-236.962,  677.988, False],
    "2": [-437.931,  665.832, False],
    "3": [-624.567,  667.941, False],
}

# orienteringsukjente første kolonne 0 og andre i A matrisen
orienteringer = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
}

# koordinatukjente
ukjente = {
    "1": [4, 5],
    "2": [6, 7],
    "3": [8, 9],
}

antall_ukjente = 10 #O_A, O_B,O_C,O_D,1x_x,1x_y,2x_x,2x_y,3x_x,3x_y OGSA 32-10= 22 overbestemmelser obs-ukjente

retning_obs = [
    # [fra, til, observert retning i gon]

    ["A", "B", 0.0000],
    ["A", "1", 52.3140],
    ["A", "2", 63.3012],
    ["A", "3", 77.7350],
    ["A", "D", 99.4937],

    ["B", "C", 0.0000],
    ["B", "1", 12.4318],
    ["B", "2", 28.0667],
    ["B", "3", 38.8933],
    ["B", "A", 88.2074],

    ["C", "D", 0.0000],
    ["C", "3", 61.4030],
    ["C", "2", 75.5923],
    ["C", "1", 93.7009],
    ["C", "B", 107.3340],

    ["D", "A", 0.0000],
    ["D", "3", 21.9039],
    ["D", "2", 37.1011],
    ["D", "1", 50.4292],
    ["D", "C", 104.9644],
]

avstand_obs = [
    # [fra, til, observert avstand i meter]

    ["A", "1", 910.582],
    ["A", "2", 770.264],
    ["A", "3", 680.854],

    ["B", "1", 718.213],
    ["B", "2", 796.945],
    ["B", "3", 914.463],

    ["C", "3", 794.154],
    ["C", "2", 711.135],
    ["C", "1", 655.737],

    ["D", "3", 676.420],
    ["D", "2", 753.490],
    ["D", "1", 863.553],
]

sigma_retning = 0.0005  # gon


def sigma_avstand(D):
    # D i meter
    return 0.003 + 0.003 * (D / 1000)


# ============================================================
# 2. HJELPEFUNKSJONER
# ============================================================

def normaliser_gon(v):
    # holder vinkler i området -200 til 200 gon
    while v > 200:
        v -= 400
    while v < -200:
        v += 400
    return v


def beregn_retning(fra_punkt, til_punkt):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    r = atan2(dY, dX) * 200 / pi

    if r < 0:
        r += 400

    return r


def beregn_avstand(fra_punkt, til_punkt):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    return sqrt(dX**2 + dY**2)


# ============================================================
# 3. STARTORIENTERINGER
# ============================================================
# Første retning fra hver stasjon brukes som referanse.
# Dette gir l = 0 for første retning fra hver stasjon.

orientering_start = {}

for fra_punkt in orienteringer:
    for obs in retning_obs:
        if obs[0] == fra_punkt:
            til_punkt = obs[1]
            obs_retning = obs[2]

            beregnet = beregn_retning(fra_punkt, til_punkt)

            orientering_start[fra_punkt] = normaliser_gon(
                beregnet - obs_retning
            )
            break


# ============================================================
# 4. OBSERVASJONSLIKNINGER
# ============================================================

def lag_retning_obs(fra_punkt, til_punkt, obs_retning):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    beregnet = beregn_retning(fra_punkt, til_punkt)
    orientering = orientering_start[fra_punkt]

    # Rapport-vennlig konvensjon:
    # l = obs + orientering - beregnet
    l = normaliser_gon(obs_retning + orientering - beregnet)

    A_rad = [0] * antall_ukjente

    # Viktig: -1 for å følge rapportens A-matrise
    A_rad[orienteringer[fra_punkt]] = -1

    # derivert for til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]

        A_rad[colX] = -(dY / D**2) * 200 / pi
        A_rad[colY] =  (dX / D**2) * 200 / pi

    # derivert for fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]

        A_rad[colX] =  (dY / D**2) * 200 / pi
        A_rad[colY] = -(dX / D**2) * 200 / pi

    # relativ vekt, slik rapporten bruker
    p = 1

    return A_rad, l, p


def lag_avstand_obs(fra_punkt, til_punkt, obs_avstand):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    # l = observert - beregnet
    l = obs_avstand - D

    A_rad = [0] * antall_ukjente

    # derivert for til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]

        A_rad[colX] = dX / D
        A_rad[colY] = dY / D

    # derivert for fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]

        A_rad[colX] = -dX / D
        A_rad[colY] = -dY / D

    sigma = sigma_avstand(D)

    # relativ vekt mot retning
    p = sigma_retning**2 / sigma**2

    return A_rad, l, p


# ============================================================
# 5. BYGG A, l OG P
# ============================================================

A_liste = []
l_liste = []
p_liste = []
navn_liste = []

for fra_punkt, til_punkt, obs_retning in retning_obs:
    A_rad, l_verdi, p_verdi = lag_retning_obs(
        fra_punkt,
        til_punkt,
        obs_retning
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Retning {fra_punkt}->{til_punkt}")

for fra_punkt, til_punkt, obs_avstand in avstand_obs:
    A_rad, l_verdi, p_verdi = lag_avstand_obs(
        fra_punkt,
        til_punkt,
        obs_avstand
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Avstand {fra_punkt}->{til_punkt}")

A = np.array(A_liste, dtype=float)
l = np.array(l_liste, dtype=float).reshape(-1, 1)
P = np.diag(p_liste)


# ============================================================
# 6. MINSTE KVADRATERS METODE
# ============================================================

N = A.T @ P @ A
h = A.T @ P @ l

dx = np.linalg.solve(N, h)

v = A @ dx - l

frihetsgrader = len(l) - antall_ukjente

s0_hat = sqrt((v.T @ P @ v)[0, 0] / frihetsgrader)

Qxx = np.linalg.inv(N)

std_parametre = s0_hat * np.sqrt(np.diag(Qxx)).reshape(-1, 1)


# ============================================================
# 7. PEN UTSKRIFT
# ============================================================

kolonnenavn = [
    "o_A",
    "o_B",
    "o_C",
    "o_D",

    "dX_1",
    "dY_1",

    "dX_2",
    "dY_2",

    "dX_3",
    "dY_3",
] #NB ENDRE PÅ EXAMENNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN

def print_matrix(navn, M, rader=None, kolonner=None, des=6):
    print("\n" + "=" * 80)
    print(navn)
    print("=" * 80)

    M = np.array(M)

    if M.ndim == 1:
        M = M.reshape(-1, 1)

    if kolonner is None:
        kolonner = [f"c{i+1}" for i in range(M.shape[1])]

    if rader is None:
        rader = [f"r{i+1}" for i in range(M.shape[0])]

    print("".ljust(24), end="")
    for k in kolonner:
        print(f"{k:>14}", end="")
    print()

    for navn_rad, row in zip(rader, M):
        print(f"{navn_rad:<24}", end="")
        for verdi in row:
            print(f"{verdi:>14.{des}f}", end="")
        print()


print("\nSTARTORIENTERINGER")
for punkt, verdi in orientering_start.items():
    print(f"{punkt}: {verdi:.6f} gon")

print_matrix("A-matrise", A, navn_liste, kolonnenavn, des=6)
print_matrix("l-vektor", l, navn_liste, ["l"], des=6)
print_matrix("P diagonal", np.diag(P), navn_liste, ["p"], des=8)

print_matrix("N = A.T P A", N, kolonnenavn, kolonnenavn, des=6)
print_matrix("h = A.T P l", h, kolonnenavn, ["h"], des=8)

print_matrix("Korreksjoner dx", dx, kolonnenavn, ["dx"], des=8)
print_matrix("Residualer v = A dx - l", v, navn_liste, ["v"], des=8)

print("\n" + "=" * 80)
print("STANDARDAVVIK TIL VEKTSENHETEN")
print("=" * 80)
print(f"s0_hat = {s0_hat:.8f}")

print_matrix("Qxx = N^-1", Qxx, kolonnenavn, kolonnenavn, des=6)
print_matrix("Standardavvik til parametere", std_parametre, kolonnenavn, ["std"], des=8)


print("\n" + "=" * 80)
print("NYE KOORDINATER")
print("=" * 80)

for punkt in ukjente:
    colX, colY = ukjente[punkt]

    X_gammel = punkter[punkt][0]
    Y_gammel = punkter[punkt][1]

    X_ny = X_gammel + dx[colX, 0]
    Y_ny = Y_gammel + dx[colY, 0]

    print(f"{punkt}:")
    print(f"  X gammel = {X_gammel:.6f}")
    print(f"  Y gammel = {Y_gammel:.6f}")
    print(f"  dX       = {dx[colX, 0]:.8f}")
    print(f"  dY       = {dx[colY, 0]:.8f}")
    print(f"  X ny     = {X_ny:.6f}")
    print(f"  Y ny     = {Y_ny:.6f}")


print("\n" + "=" * 80)
print("ORIENTERINGSKORREKSJONER")
print("=" * 80)

for punkt in orienteringer:
    colO = orienteringer[punkt]

    print(f"{punkt}: do = {dx[colO, 0]:.8f} gon")